# MedSIGHT — Segmentation Demo

MedSIGHT can emit segmentation masks together with its text answer by
enabling `output_segmentation: true` in `configs/model.yaml`. This notebook
shows how to:

1. Load the model in segmentation mode.
2. Ask it to segment a region in a medical image and obtain `mask_logits`.
3. Visualize the mask overlay on the original image.

Make sure the model checkpoint you point to was trained with the segmentation
head; using a plain VQA checkpoint with `output_segmentation: true` will fail
to load.

In [ ]:
import os, sys
REPO_ROOT = os.environ.get('MEDSIGHT_REPO', '/path/to/RegTok/RegLLM')
sys.path.insert(0, REPO_ROOT)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

from llava.eval.chatbot import RegLLMChatbot

MODEL_CONFIG = os.path.join(REPO_ROOT, 'llava/eval/configs/model.yaml')
bot = RegLLMChatbot.from_config(MODEL_CONFIG, device='cuda')
assert bot.model_args.output_segmentation, 'Set output_segmentation: true in model.yaml'

## 1. Generate a segmentation mask

`bot.inference(..., output_seg=True)` returns a dict with the decoded text and
the raw mask logits `(B, K, H, W)`. `K` is the number of segmentation tokens
emitted by the model — typically 1 per requested region.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image_path = '/path/to/sample/source.jpg'
prompt     = 'Please segment the kidney.'

result = bot.inference(prompt, image_path, output_seg=True)
print('Answer:', result['answers'][0])

seg_logits = result['mask_logits']
print('mask_logits:', seg_logits.shape)

## 2. Visualize the mask overlay

In [ ]:
import numpy as np

image = Image.open(image_path).convert('RGB')
mask  = seg_logits[0, 0].float().detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image);                 axes[0].set_title('Input');   axes[0].axis('off')
axes[1].imshow(mask, cmap='jet');      axes[1].set_title('Mask');    axes[1].axis('off')
axes[2].imshow(image)
axes[2].imshow(mask, cmap='jet', alpha=0.5)
axes[2].set_title('Overlay');          axes[2].axis('off')
plt.tight_layout(); plt.show()

## 3. Multi-region prompts

Asking for several anatomical regions in one prompt makes the model emit
multiple segmentation tokens; each one becomes a separate channel in
`mask_logits`.

In [ ]:
result = bot.inference('Please segment all the organs.', image_path, output_seg=True)
print('Answer:', result['answers'][0])
print('mask_logits:', result['mask_logits'].shape)

K = result['mask_logits'].shape[1]
fig, axes = plt.subplots(1, K, figsize=(3 * K, 3))
if K == 1:
    axes = [axes]
for k in range(K):
    axes[k].imshow(result['mask_logits'][0, k].float().detach().cpu().numpy(), cmap='jet')
    axes[k].set_title(f'Region {k}'); axes[k].axis('off')
plt.tight_layout(); plt.show()